# Day 39 — Twenty Questions

## Decision Trees

**Boston University · Data Science Program**

---

Yesterday k-NN told you *"denied, because the 15 students most like you were mostly denied."*
True, but useless in a letter to a student.

Today's model answers differently:

> *"Denied, because your marks were 68, you play no sport, and your attendance was below 72."*

Three sentences, each one checkable by a human. That is a **decision tree**, and it is
the first model in this course you can read out loud.

### What you will be able to do by the end

| | |
|---|---|
| 1 | Read a trained tree and explain any single prediction |
| 2 | Compute Gini impurity by hand and say why a split was chosen |
| 3 | Spot an overfitted tree from its train/test gap |
| 4 | Prune it with `max_depth` and `min_samples_leaf`, tuned by cross-validation |
| 5 | Say when a tree is the right model and when it is the wrong one |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, cross_validate, GridSearchCV)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

CV = StratifiedKFold(5, shuffle=True, random_state=0)
print("ready")

---
# 0 · The data

A scholarship committee has been awarding scholarships for years. We have **900 of their
past decisions**. Nobody wrote the rule down — we are going to recover it from the decisions.

Four columns about each student, and whether they got the award.

In [ ]:
rng = np.random.default_rng(3900)
N = 900

marks         = rng.normal(70, 14, N).clip(35, 100).round(0)
family_income = rng.normal(380000, 210000, N).clip(40000, 1500000).round(0)
attendance    = rng.normal(82, 12, N).clip(40, 100).round(0)
sport         = rng.binomial(1, 0.25, N).astype(float)

# the committee's actual (unwritten) rule
rule = (((marks >= 75) & (attendance >= 80))
        | (marks >= 88)
        | ((sport == 1) & (marks >= 62) & (attendance >= 75)))
rule = rule & (family_income <= 700000)

# ...applied by humans, who are not perfectly consistent: 8% of decisions go the other way
flip = rng.uniform(0, 1, N) < 0.08
awarded = np.where(flip, 1 - rule.astype(int), rule.astype(int))

sch = pd.DataFrame({"marks": marks, "family_income": family_income,
                    "attendance": attendance, "sport": sport, "awarded": awarded})

X = sch[["marks", "family_income", "attendance", "sport"]]
y = sch["awarded"]

print(sch.head())
print()
print("rows:         ", len(sch))
print("awarded:      ", int(y.sum()), f"({y.mean():.4f})")
print("BASELINE — always guess 'not awarded':", round(1 - y.mean(), 4))

**Write the baseline down.** `0.6678`. A model that scores below that is worse than a
rule you could write on a napkin, and no amount of sophistication rescues it.

---
# 1 · Train one and read it

Three lines. Notice what is *absent*: no `StandardScaler`, no encoding, no `k` to pick.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X, y)

print(export_text(tree, feature_names=list(X.columns), decimals=0))

Read that top to bottom and it is just nested if-statements. `|---` marks a question;
indentation marks depth; `class:` is the answer at a leaf.

`random_state=0` matters. When two questions score *exactly* the same, sklearn picks
between them at random. Set the seed or the tree moves between runs.

### The same thing, drawn

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(tree, feature_names=list(X.columns),
          class_names=["not awarded", "awarded"],
          filled=True, rounded=True, fontsize=9, impurity=True, ax=ax)
ax.set_title("the depth-3 tree sklearn built from 900 decisions", fontsize=13)
plt.tight_layout()
plt.show()

Each box shows four things:

| line | meaning |
|---|---|
| `marks <= 74.5` | the question. Students answering **yes** go **left** |
| `gini = 0.444` | how mixed this group is. 0 = all one class, 0.5 = even mix |
| `samples = 900` | how many training students are standing here |
| `value = [601, 299]` | how many of them were `[not awarded, awarded]` |

A box with no question is a **leaf** — that is where a prediction comes from.

## 1.1 · Walk one student down by hand

Let us predict for a student with marks 91, income 250000, attendance 84, no sport.

In [ ]:
student = pd.DataFrame([{"marks": 91.0, "family_income": 250000.0,
                         "attendance": 84.0, "sport": 0.0}])

print("Q1  marks <= 74.5 ?        91 > 74.5           -> NO,  go right")
print("Q2  attendance <= 78.5 ?   84 > 78.5           -> NO,  go right")
print("Q3  income <= 704710 ?     250000 <= 704710    -> YES, go left")
print()
print("we land in a leaf holding 13 'not awarded' and 158 'awarded' students")
print(f"so the probability is 158 / 171 = {158/171:.4f}")
print()
print("sklearn agrees:")
print("  predict      ", tree.predict(student)[0])
print("  predict_proba", tree.predict_proba(student)[0].round(4))

**That is the whole prediction mechanism.** No coefficients multiplied, no distances
measured. Answer the questions, land in a leaf, read the majority off it.

`predict_proba` is literally the fraction of training students in that leaf who were
awarded — which is also why a leaf holding 3 students gives you a confident-looking
probability you should not trust.

Let's confirm the leaf really does hold 171 students.

In [ ]:
leaf_id = tree.apply(student)[0]          # which leaf did this student land in?
rows    = tree.apply(X) == leaf_id        # which training rows share it?

print("leaf id:", leaf_id)
print("training students in that leaf:", rows.sum())
print(y[rows].value_counts().rename({0: "not awarded", 1: "awarded"}))
print()
print("fraction awarded:", round(y[rows].mean(), 4))

---
# 2 · Why scaling does not matter

This is the sharpest contrast with Day 38, so it is worth proving rather than asserting.

A tree only ever asks *"is this value above or below some number?"* Multiply a column by
1000 and the best threshold multiplies by 1000 too. **The same students go the same way.**

k-NN, by contrast, adds up squared differences across columns, so a column measured in
hundreds of thousands drowns out a column measured in tens.

In [ ]:
X_lakhs = X.copy(); X_lakhs["family_income"] = X_lakhs["family_income"] / 100000
X_big   = X.copy(); X_big["marks"]           = X_big["marks"] * 1000

def cv_score(model, data):
    return cross_val_score(model, data, y, cv=CV).mean()

mk_tree = lambda: DecisionTreeClassifier(max_depth=4, random_state=0)
mk_knn  = lambda: KNeighborsClassifier(15)

rows = []
for name, data in [("as it comes", X), ("income in lakhs", X_lakhs), ("marks x 1000", X_big)]:
    rows.append({"version": name,
                 "tree": round(cv_score(mk_tree(), data), 4),
                 "k-NN": round(cv_score(mk_knn(),  data), 4)})
rows.append({"version": "standard-scaled",
             "tree": round(cv_score(make_pipeline(StandardScaler(), mk_tree()), X), 4),
             "k-NN": round(cv_score(make_pipeline(StandardScaler(), mk_knn()),  X), 4)})

print(pd.DataFrame(rows).to_string(index=False))

The tree column is **0.8967 four times over** — not approximately, exactly. The k-NN
column swings from 0.6711 to 0.8622.

> **Rule of thumb.** Distance-based models (k-NN, SVM, k-means) need scaling.
> Split-based models (trees, forests, boosting) do not. Scaling a tree is harmless,
> just pointless.

---
# 3 · How the tree chose that question

Everything so far has been *reading* a tree. Now, how was it built?

There is one idea: **measure how mixed a group is, then try every possible question and
keep whichever makes the two resulting groups least mixed.**

## 3.1 · Gini impurity

For a group where a fraction `p` were awarded:

$$\text{Gini} = 1 - p^2 - (1-p)^2$$

- all one class → `p` is 0 or 1 → Gini **0.0** (perfectly pure)
- an even mix → `p = 0.5` → Gini **0.5** (as messy as it gets)

In [ ]:
def gini(labels):
    labels = np.asarray(labels)
    if len(labels) == 0:
        return 0.0
    p = labels.mean()
    return 1 - p**2 - (1 - p)**2

for name, p in [("all denied",   0.0),
                ("9 in 10 denied", 0.1),
                ("2 in 3 denied",  1/3),
                ("even mix",      0.5)]:
    print(f"  {name:<16} p={p:.3f}   gini = {1 - p**2 - (1-p)**2:.4f}")

print()
print(f"our 900 students: p = {y.mean():.4f}, gini = {gini(y):.4f}")

In [ ]:
p = np.linspace(0, 1, 300)
plt.figure(figsize=(8, 4))
plt.plot(p, 1 - p**2 - (1-p)**2, lw=3, label="Gini")
plt.plot(y.mean(), gini(y), "D", ms=11, color="#F0A530", zorder=5)
plt.annotate(f"our data\np={y.mean():.4f}", (y.mean(), gini(y)),
             xytext=(0.12, 0.25), fontsize=11, fontweight="bold",
             arrowprops=dict(arrowstyle="->", lw=1.8))
plt.xlabel("p = fraction awarded"); plt.ylabel("Gini impurity")
plt.title("impurity is highest at an even mix, zero at either extreme")
plt.tight_layout(); plt.show()

## 3.2 · Scoring one split

Take the question `marks <= 74.5`. Split the 900 students, measure each side, and
**weight by how many students landed there** — otherwise the tree would adore a split
that makes one perfectly pure group of three.

In [ ]:
mask = X["marks"] <= 74.5
left, right = y[mask], y[~mask]

g_before = gini(y)
g_left, g_right = gini(left), gini(right)
weighted = (len(left) * g_left + len(right) * g_right) / len(y)

print(f"before the split : {len(y):>3} students, gini {g_before:.4f}")
print(f"  LEFT  (<=74.5) : {len(left):>3} students, gini {g_left:.4f}")
print(f"  RIGHT ( >74.5) : {len(right):>3} students, gini {g_right:.4f}")
print()
print(f"weighted gini after = ({len(left)}/900) x {g_left:.4f} "
      f"+ ({len(right)}/900) x {g_right:.4f} = {weighted:.4f}")
print(f"improvement         = {g_before:.4f} - {weighted:.4f} = {g_before - weighted:.4f}")

## 3.3 · Scoring every other question

Now the same calculation for a handful of alternatives. The tree keeps whichever one
drops impurity the most.

In [ ]:
def score_split(col, threshold):
    m = X[col] <= threshold
    if m.sum() == 0 or (~m).sum() == 0:
        return np.nan
    return (m.sum() * gini(y[m]) + (~m).sum() * gini(y[~m])) / len(y)

candidates = [("marks", 74.5), ("marks", 60.5), ("sport", 0.5),
              ("attendance", 79.5), ("family_income", 700000.5)]

out = [{"question": f"{c} <= {t:g}",
        "weighted gini": round(score_split(c, t), 4),
        "impurity drop": round(gini(y) - score_split(c, t), 4)}
       for c, t in candidates]

print(f"before any split: {gini(y):.4f}\n")
print(pd.DataFrame(out).sort_values("impurity drop", ascending=False).to_string(index=False))

`marks <= 74.5` wins by a wide margin — and that is exactly the question sklearn put at
the root. We just reproduced its decision by hand.

## 3.4 · "Every question" really means every question

Those five were a sample. The tree tries **every midpoint between consecutive distinct
values, in every column.**

In [ ]:
total = 0
for col in X.columns:
    n = len(np.unique(X[col])) - 1
    total += n
    print(f"  {col:<15} {n:>4} candidate thresholds")
print(f"  {'TOTAL':<15} {total:>4}  — all scored, just to choose the root")

In [ ]:
# the full scan over one column, drawn
vals  = np.unique(X["marks"])
mids  = (vals[:-1] + vals[1:]) / 2
scores = np.array([score_split("marks", t) for t in mids])
best = mids[np.nanargmin(scores)]

plt.figure(figsize=(9, 4))
plt.plot(mids, scores, "o-", ms=4, lw=2)
plt.plot(best, np.nanmin(scores), "o", ms=13, color="#1F9E86", zorder=5)
plt.annotate(f"winner: marks <= {best}\ngini {np.nanmin(scores):.4f}",
             (best, np.nanmin(scores)), xytext=(best - 24, np.nanmin(scores) + 0.022),
             fontsize=11, fontweight="bold",
             arrowprops=dict(arrowstyle="->", lw=2, color="#1F9E86"))
plt.xlabel("threshold tried on marks"); plt.ylabel("weighted Gini after")
plt.title(f"all {len(mids)} thresholds on the marks column, scored one by one")
plt.tight_layout(); plt.show()

print("sklearn's root:", X.columns[tree.tree_.feature[0]], "<=", tree.tree_.threshold[0])

## 3.5 · Then it does the whole thing again

That was only the **root**. The tree now takes the 592 students on the left, forgets the
rest of the world exists, and repeats the entire search inside that group. Then the same
for the 308 on the right. Then inside each of *those* halves. That is the recursion.

**Four steps, repeated:**

1. **Measure** the Gini impurity of the group you are standing in
2. **Try** all candidate questions, keep the biggest impurity drop
3. **Split** the rows left and right by their answer
4. **Repeat**, separately, inside each half

It stops when a group is pure, or when a rule you set says stop. That rule is section 4.

> The tree is **greedy**: it takes the best question available *right now* and never goes
> back to reconsider. A different first question might have led somewhere better overall,
> but it will never find out. Greedy is fast, and usually good enough.

### Gini or entropy?

Entropy, $-p\log_2 p - (1-p)\log_2(1-p)$, is the alternative impurity measure.
Different scale, same shape, nearly always the same tree.

In [ ]:
p = y.mean()
print(f"at our root:  gini = {gini(y):.4f}    entropy = {-(p*np.log2(p) + (1-p)*np.log2(1-p)):.4f}")
print()
for crit in ["gini", "entropy"]:
    s = cross_val_score(DecisionTreeClassifier(criterion=crit, max_depth=4, random_state=0),
                        X, y, cv=CV).mean()
    print(f"  criterion={crit:<8} CV {s:.4f}")
print("\ndifference: 0.0089 — smaller than the noise between folds. Use the default.")

---
# 4 · Left alone, a tree memorises

Here is the trap. Remove `max_depth` and the tree keeps splitting until every leaf is
pure — which it can always achieve, by making a leaf for practically every student.

In [ ]:
res = cross_validate(DecisionTreeClassifier(random_state=0), X, y,
                     cv=CV, return_train_score=True)
full = DecisionTreeClassifier(random_state=0).fit(X, y)

print(f"training accuracy      {res['train_score'].mean():.4f}   <- perfect")
print(f"cross-validated        {res['test_score'].mean():.4f}   <- the honest number")
print(f"gap                   {res['train_score'].mean() - res['test_score'].mean():+.4f}")
print(f"leaves                 {full.get_n_leaves()}  for 900 students")
print(f"depth                  {full.get_depth()}")

100% on the training rows, because it wrote down an answer for each one. That is not
learning the committee's rule — it is copying the committee's homework.

**A gap of +0.1522 between training and cross-validation is the signature of
overfitting.** You met it on Day 36; here it is again, much larger.

## 4.1 · Knob one — `max_depth`

In [ ]:
rows = []
for d in [1, 2, 3, 4, 5, 6, 8, 10, 12, None]:
    r = cross_validate(DecisionTreeClassifier(max_depth=d, random_state=0),
                       X, y, cv=CV, return_train_score=True)
    leaves = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X, y).get_n_leaves()
    rows.append({"max_depth": "none" if d is None else d, "leaves": leaves,
                 "train": round(r["train_score"].mean(), 4),
                 "CV":    round(r["test_score"].mean(), 4),
                 "gap":   round(r["train_score"].mean() - r["test_score"].mean(), 4)})

depth_tbl = pd.DataFrame(rows)
print(depth_tbl.to_string(index=False))

In [ ]:
d = depth_tbl[depth_tbl.max_depth != "none"].copy()
d["max_depth"] = d["max_depth"].astype(int)
plt.figure(figsize=(9, 4.5))
plt.plot(d.max_depth, d.train, "o-", lw=2.5, label="training")
plt.plot(d.max_depth, d.CV,    "o-", lw=2.5, label="cross-validated")
plt.fill_between(d.max_depth, d.CV, d.train, alpha=0.15, color="red")
best = d.loc[d.CV.idxmax()]
plt.axvline(best.max_depth, ls=":", lw=2)
plt.annotate(f"best: depth {best.max_depth}, CV {best.CV}",
             (best.max_depth, best.CV), xytext=(best.max_depth + 1.2, best.CV - 0.06),
             fontsize=11, fontweight="bold", arrowprops=dict(arrowstyle="->", lw=1.8))
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.legend()
plt.title("the two lines pulling apart is your warning light")
plt.tight_layout(); plt.show()

Shallow trees **underfit** — depth 1 is a single question and scores 0.7722. Deep trees
**memorise**. The best cross-validated depth here is **5**, at **0.9078**.

## 4.2 · Knob two — `min_samples_leaf`

A leaf built from 2 students is a coincidence, not a rule. This knob forbids them.

In [ ]:
rows = []
for m in [1, 2, 5, 10, 20, 50, 100]:
    r = cross_validate(DecisionTreeClassifier(min_samples_leaf=m, random_state=0),
                       X, y, cv=CV, return_train_score=True)
    leaves = DecisionTreeClassifier(min_samples_leaf=m, random_state=0).fit(X, y).get_n_leaves()
    rows.append({"min_samples_leaf": m, "leaves": leaves,
                 "train": round(r["train_score"].mean(), 4),
                 "CV":    round(r["test_score"].mean(), 4),
                 "gap":   round(r["train_score"].mean() - r["test_score"].mean(), 4)})
print(pd.DataFrame(rows).to_string(index=False))

Watch the `gap` column collapse from **+0.1522** at `min_samples_leaf=1` to **+0.0094**
at 10. Push it to 100 and the tree underfits instead (CV 0.8122) — there is a sweet spot,
and you find it by searching, not by guessing.

Other knobs you will see: `min_samples_split`, `max_leaf_nodes`, and `ccp_alpha`
(cost-complexity pruning, which grows the full tree then snips branches back). They all
do the same job: stop the tree from carving the data into individual students.

## 4.3 · Tune them properly

In [ ]:
grid = GridSearchCV(
    DecisionTreeClassifier(random_state=0),
    {"max_depth": [2, 3, 4, 5, 6, 8, None],
     "min_samples_leaf": [1, 5, 10, 20, 50]},
    cv=CV, n_jobs=-1)
grid.fit(X, y)

print("best parameters:", grid.best_params_)
print("best CV score:  ", round(grid.best_score_, 4))
print()
print("against:")
print("  baseline (guess the majority)", round(1 - y.mean(), 4))
print("  no limits at all             ", round(res["test_score"].mean(), 4))

35 combinations × 5 folds = 175 trees fitted, and the winner is chosen on the
**cross-validated** score — never the training score.

---
# 5 · Which column mattered?

A tree keeps a running total of how much impurity each column removed, weighted by how
many students passed through each split. That is `feature_importances_`.

In [ ]:
best_tree = DecisionTreeClassifier(**grid.best_params_, random_state=0).fit(X, y)

imp = (pd.Series(best_tree.feature_importances_, index=X.columns)
         .sort_values(ascending=False))
print(imp.round(4))
print("\nthey sum to", round(imp.sum(), 4))

imp.sort_values().plot.barh(figsize=(8, 3.4), color="#8268B5")
plt.xlabel("feature importance"); plt.title("how much impurity each column removed")
plt.tight_layout(); plt.show()

Read this carefully, because it is easy to over-read:

- it says **how useful a column was to this particular tree**, not how important it is
  in the world
- two columns carrying the same information will **split the credit**, making both look
  weak
- a column with many distinct values (like `family_income`) gets more chances to be
  chosen, which biases it **upward**

It is a hint about where to look, not a measurement of cause.

---
# 6 · When is a tree the right choice?

## 6.1 · The shape of the boundary

Let us use just two columns so we can draw the decision boundary, and compare three
models on identical data.

In [ ]:
X2 = sch[["marks", "attendance"]]
mg, ag = np.meshgrid(np.linspace(35, 100, 300), np.linspace(40, 100, 300))
grid_pts = pd.DataFrame(np.c_[mg.ravel(), ag.ravel()], columns=X2.columns)

models = [("decision tree (depth 4)", DecisionTreeClassifier(max_depth=4, random_state=0)),
          ("logistic regression",     make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
          ("k-NN (k=15)",             make_pipeline(StandardScaler(), KNeighborsClassifier(15)))]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
samp = np.random.default_rng(0).choice(len(X2), 320, replace=False)
for ax, (name, model) in zip(axes, models):
    model.fit(X2, y)
    z = np.asarray(model.predict(grid_pts)).reshape(mg.shape)
    ax.contourf(mg, ag, z, levels=[-0.5, 0.5, 1.5], colors=["#FCEBE7", "#E6F5F1"])
    ax.contour(mg, ag, z, levels=[0.5], colors=["#131F33"], linewidths=2)
    ax.scatter(X2.values[samp, 0], X2.values[samp, 1],
               c=["#1F9E86" if v else "#E05C45" for v in y.values[samp]],
               s=12, alpha=0.75)
    ax.set_title(f"{name}\nCV {cross_val_score(model, X2, y, cv=CV).mean():.4f}", fontsize=11)
    ax.set_xlabel("marks")
axes[0].set_ylabel("attendance")
plt.tight_layout(); plt.show()

Look at the shapes, not just the scores.

The committee's real rule contains *"marks at least 75 **AND** attendance at least 80"*.
That is a **corner**. The tree draws corners for free, because every split is a
horizontal or vertical cut. Logistic regression can only draw one straight line, so it
has to slice diagonally through the corner and gets it wrong on both sides.

This is also the tree's weakness in reverse: if the truth really were a smooth diagonal,
the tree would approximate it with an ugly staircase and lose.

## 6.2 · All four, on the full data

In [ ]:
contenders = {
    "baseline (majority)": None,
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "k-NN (k=15)":         make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
    "tree (depth 4)":      DecisionTreeClassifier(max_depth=4, random_state=0),
    "tree (tuned)":        DecisionTreeClassifier(**grid.best_params_, random_state=0),
}
for name, model in contenders.items():
    score = 1 - y.mean() if model is None else cross_val_score(model, X, y, cv=CV).mean()
    print(f"  {name:<22} {score:.4f}")

The tree beats logistic regression by nearly 10 points **on this data** — because this
truth is made of thresholds. Change the problem and the ranking changes. On Day 35's
data, logistic regression would win comfortably.

> There is no best model. There is only the model whose shape matches the truth,
> and cross-validation is how you find out which one that is.

## 6.3 · The honest weakness: instability

Trees are twitchy. Drop a few rows and a split can move, because the top two candidate
questions are often nearly tied — and the winner takes the whole branch.

In [ ]:
base = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
print("full data, first three questions:")
print("  ", [f"{X.columns[base.tree_.feature[k]]}<={base.tree_.threshold[k]:.1f}"
             for k in range(base.tree_.node_count) if base.tree_.feature[k] >= 0][:3])
print()
print("now refit on a random 90% of the rows, five times:")
for s in range(5):
    idx = np.random.default_rng(s).choice(len(X), int(0.9 * len(X)), replace=False)
    t = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X.iloc[idx], y.iloc[idx])
    qs = [f"{X.columns[t.tree_.feature[k]]}<={t.tree_.threshold[k]:.1f}"
          for k in range(t.tree_.node_count) if t.tree_.feature[k] >= 0][:3]
    print(f"  sample {s}: {qs}")

The root held all five times — it won by a wide margin, so it is safe. But a **third-level
question moved in one of the five samples**. Deeper splits see fewer students, so the
margins between candidates are thinner and the winner flips more easily.

That is the honest picture: not "trees are chaotic", but "the deeper you read, the less
you should trust the exact threshold".

And it points straight at tomorrow's fix: if one tree is twitchy, grow **many** trees on
different slices of the data and let them vote. The twitches cancel out. That is a
**random forest**.

---
# 7 · Summary

| | |
|---|---|
| **What a tree is** | A flowchart of yes/no questions on one column at a time. A leaf holds past students; you read the majority off it. |
| **How it learns** | Score every candidate question by Gini impurity, keep the biggest drop, repeat inside each half. Greedy, never reconsiders. |
| **Scaling** | Irrelevant. `0.8967` raw, scaled, and with income in lakhs — identical. The exact opposite of k-NN. |
| **Overfitting** | Unlimited depth → train `1.0000`, CV `0.8478`, 137 leaves. The train/CV gap is the warning light. |
| **Pruning** | `max_depth` and `min_samples_leaf`. Tuned here to CV `0.9078` against a `0.6678` baseline. |
| **When to use one** | Thresholds, AND-rules, mixed column types, and when you must explain the decision. |
| **When not to** | Smooth trends, and whenever you need the best possible accuracy from one model. |

### The numbers from today

| quantity | value |
|---|---|
| baseline (always "not awarded") | 0.6678 |
| root Gini impurity | 0.4437 |
| best root split | `marks <= 74.5`, impurity drop 0.1068 |
| candidate splits scored for the root | 960 |
| unpruned tree: train / CV / leaves | 1.0000 / 0.8478 / 137 |
| tuned tree (max_depth=5) | 0.9078 |
| logistic regression / k-NN | 0.8100 / 0.8622 |

---

### Next: Day 40 — Random Forests

One tree is readable but twitchy. A hundred trees, each grown on a different slice of the
data and a different subset of the columns, voting together, is neither — but it is a lot
more accurate. We will see exactly how much of today's instability disappears.